# Initial Set Up

## Imports

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os 
import csv
import sys
import csv
import re

sys.path.append("../src")

import matplotlib as mpl
import matplotlib.pyplot as plot
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_regression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from collections import Counter, defaultdict
from tqdm import tqdm

from bisect import bisect_left, bisect_right

from global_utils.graphs_utils import *
from global_utils.files_utils import clean_csv
from data_exploration.extraction_utils import get_transactions_from_raw_log, assign_correct_values

## Files

In [ ]:
DATABASE_DIR = "../Database"
if os.path.exists(DATABASE_DIR):
    print("Can see Data_Dir")
DATA_DIR = os.path.join(DATABASE_DIR, "CriticalChargingSessions")
DATA_DIR_CLEAN = os.path.join(DATABASE_DIR, "CriticalChargingSessions/Clean")
DATA_DIR = os.path.join(DATABASE_DIR, "CriticalChargingSessions")
DATA_DIR_CLEAN = os.path.join(DATABASE_DIR, "CriticalChargingSessions/Clean")
DATA_DIR_BIG = os.path.join(DATABASE_DIR, "data_exploded/TroubleshootingChargerExplodedPlugs")
DATA_DIR_BIG_CLEAN = os.path.join(DATABASE_DIR, "data_exploded/TroubleshootingChargerExplodedPlugs/Clean")
DATA_DIR_EXTENDED = os.path.join(DATABASE_DIR, "CriticalChargingSessions_Bigger")
DATA_DIR_EXTENDED_CLEAN = os.path.join(DATABASE_DIR, "CriticalChargingSessions_Bigger/Clean")

SAVE_PATH_MANUAL = "../../Images"
plot.rcParams['savefig.format'] = "svg"
plot.rcParams['figure.figsize'] = [6.6, 5.0]

FILES = [file_ for file_ in os.listdir(DATA_DIR) if (".txt" not in file_) &  (not "Clean" in file_)]
FILES_BIG = [file_ for file_ in os.listdir(DATA_DIR_BIG) if (".txt" not in file_) &  (not "Clean" in file_) & (not "top_50_charging" in file_)]
FILES_EXTENDED = [file_ for file_ in os.listdir(DATA_DIR_EXTENDED) if (".txt" not in file_) &  (not "Clean" in file_)]

#### Fixing Files

In [ ]:
# clean_csv(DATA_DIR, DATA_DIR_CLEAN, FILES)
# clean_csv(DATA_DIR_BIG, DATA_DIR_BIG_CLEAN, FILES_BIG)
# clean_csv(DATA_DIR_EXTENDED, DATA_DIR_EXTENDED_CLEAN, FILES_EXTENDED)

### Read

In [ ]:
CLEAN_FILES = os.listdir(DATA_DIR_CLEAN)
CLEAN_FILES_BIG = os.listdir(DATA_DIR_BIG_CLEAN)
CLEAN_FILES_EXTENDED = os.listdir(DATA_DIR_EXTENDED_CLEAN)

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR_CLEAN, CLEAN_FILES[0]))
data["@timestamp"] = pd.to_datetime(data["@timestamp"])
# data["@timestamp"] = data["@timestamp"].dt.tz_localize('UTC').dt.tz_convert('UTC+05:30')
data = data.sort_values(by="@timestamp", ascending=True)

In [ ]:
# Logs containing 2hrs before explosion and 5 min afterwards.
last_time = list(data["@timestamp"])[-1]
print(last_time)
approximate_time_of_explosion = last_time - pd.Timedelta('5m')
print(approximate_time_of_explosion)

# Check the messages prior to this event:
search_time = pd.Timedelta('3m')

messages_close_to_explosion = []

for row in data.iterrows():
    if np.abs(row[1]["@timestamp"]-approximate_time_of_explosion) < search_time:
        messages_close_to_explosion.append(row[1])
dataset_close_to_explosion = pd.DataFrame(messages_close_to_explosion)

In [ ]:
datasets_close_to_explosion = [] # Dataset containing the messages that are logged within 3m of the explosion

times_of_explosion = [] # Estimated times of explosion

for element in CLEAN_FILES:
    data = pd.read_csv(os.path.join(DATA_DIR_CLEAN, element))
    data["@timestamp"] = pd.to_datetime(data["@timestamp"])
    # data["@timestamp"] = data["@timestamp"].dt.tz_localize('UTC').dt.tz_convert('UTC+05:30')
    data = data.sort_values(by="@timestamp", ascending=True)

    last_time = list(data["@timestamp"])[-1]
    approximate_time_of_explosion = last_time - pd.Timedelta('5m')

    # Check the messages prior to this event:
    search_time = pd.Timedelta('3m')

    messages_close_to_explosion = []

    for row in data.iterrows():
        if np.abs(row[1]["@timestamp"]-approximate_time_of_explosion) < search_time:
            messages_close_to_explosion.append(row[1]["@message"])
    
    datasets_close_to_explosion.append(messages_close_to_explosion)
    times_of_explosion.append(approximate_time_of_explosion)


# Analysis

## Inspect Errors

In [ ]:
for file_index in range(4):

    data = pd.read_csv(os.path.join(DATA_DIR_CLEAN, CLEAN_FILES[file_index]))
    data["@timestamp"] = pd.to_datetime(data["@timestamp"])
    data = data.sort_values(by="@timestamp", ascending=True)

    for row in data.iterrows():
        current_message = row[1]["@message"]
        # if "LCU_DC_ID" in current_message:
        # if "_codes" in current_message:
        # if r"[local0:err]" in current_message:
        if "PLC-Error" in current_message:
            print(current_message, "\n")


## Naive Comparisson

In [ ]:
# Naive comparisson
message_comparissons = []
for i in range(len(datasets_close_to_explosion)):
    message_comparissons.append(np.zeros(len(datasets_close_to_explosion[i])))

for in_dataset in range(len(datasets_close_to_explosion)): # From a first dataset
    for in_line_index, in_line in enumerate(datasets_close_to_explosion[in_dataset]):
        for out_dataset in range(in_dataset+1, len(datasets_close_to_explosion)): # Compare with the message in the others:
            for out_line_index, out_line in enumerate(datasets_close_to_explosion[out_dataset]):
                if in_line[33:] == out_line[33:]:
                    message_comparissons[in_dataset][in_line_index] += 1
                    message_comparissons[out_dataset][out_line_index] += 1

In [ ]:
repeated_messages = [(i, j, int(message_comparissons[i][j])) for i in range(len(message_comparissons)) for j in range(len(message_comparissons[i])) if message_comparissons[i][j] != 0]

In [ ]:
repeated_messages = sorted(repeated_messages, key=lambda x:x[2], reverse=True)
for i,j, z in repeated_messages:
    pass
    # print(i, j, datasets_close_to_explosion[i][j], "appeared ", z ," times ", "Time of Explosion: ", times_of_explosion[i], " diff: ", pd.Timestamp(datasets_close_to_explosion[i][j][:26])-times_of_explosion[i])



## Get Instances from Logs

In [ ]:
# Get interesting messages from logs:

# start_charge = "\"P05_START_CHARGE\" --> \"P06_CHARGE\""
# end_charge = "\"P06_CHARGE\" --> \"P00_UNKNOWN\""




session_change_message = "\"type\":"
session_change_info = {}

insulation_resistance_message = "insulation resistance" 
insulation_resistance = {}

capacitance_message = "leakage capacitance" 
capacitance = {}

meter_values_message = "MeterValues"
meter_values = {}

evse_values_message  = "EVSE_MaxPwr"
evse_values  = {}

temperature_message = "[CTD]"
temperature_values = {}

error_message = "PLC-Error-Flag"
error_values = {}


send_command_message_1 = "OCPP: Send command:"
send_command_message_2 = "Heartbeat"

session_change_keys = ["start_charging", "stop_charging"]
for file_index in range(4):

    # Each transaction may happen in either outlet 1 or outlet 2 
    #   file_index will specify which charger we are looking at. Another index must be used to identify the outlet.

    session_change_info[file_index] = {key: {1: [], 2: []} for key in session_change_keys} # Saves the information of the beginning and end of the charge.
    error_values[file_index] = {0: defaultdict(list), 1: defaultdict(list), 2: defaultdict(list)}

    insulation_resistance[file_index] = {1: [], 2: []}
    capacitance[file_index] = {1: [], 2: []}

    meter_values[file_index] = {1: [], 2: []}
    evse_values[file_index] = {1: [], 2: []}

    temperature_values[file_index] = {1: [], 2: []}

    data = pd.read_csv(os.path.join(DATA_DIR_CLEAN, CLEAN_FILES[file_index]))
    data["@timestamp"] = pd.to_datetime(data["@timestamp"])
    data = data.sort_values(by="@timestamp", ascending=True)

    for row in data.iterrows():
        current_message = row[1]["@message"]
        match = (re.search(r'DC(\d)', current_message)) # Noticed that whenever we have a DCX in a message, the X corresponds to the outlet ID. **Not all messages have DCX keyword**
        if match:
            match = int(match.group(1))

        if "OCPP: Send command:" in current_message and not "Heartbeat" in current_message:
            # print(current_message)
            value = re.search(r'OCCP: Send command:\s*(\w+)', current_message) # This is still on-going research. These Send Commands may be useful to identify certain periods during transaction.
            # print(value)
            pass

        if session_change_message in current_message:
            match = re.search(r'(\{.*\})', current_message) # Match everything inside {}
            session_change = json.loads(match.group(1))     # This corresponds to dictionary, they have information regarding the start/stop charge and transactions.
            if session_change["type"] in session_change_keys:
                match = int(session_change["outlet"]) # Get the outlet ID in case the previous match did not work.
                session_change_info[file_index][session_change["type"]][match].append(row[1]["@timestamp"])
                # file_index == Which file we are looking
                # session_change["type"] if we are starting or ending a transaction
                # match == The outlet ID
            
        elif insulation_resistance_message in current_message:
            value = (re.search(r'(\d+\.?\d*)\s*kOhm', current_message).group(1)) # Gets the numeric values that precede 'kOhm'
            if "PLC-Error-Flag" in current_message: # There are these errors, they also have resistance information, in this implementation I am also considering these values
                match = int(re.search(r'IMD_BC(\d)', current_message).group(1))
            insulation_resistance[file_index][match].append([row[1]["@timestamp"], float(value)])

        elif capacitance_message in current_message:
            value = (re.search(r'(\d+\.?\d*)\s*microF', current_message).group(1)) # Capcitance is obtained in a similar manner as Resistance, only now the key word is 'microF'.
            capacitance[file_index][match].append([row[1]["@timestamp"], float(value)])
        
        elif meter_values_message in current_message:
            value = re.search(r'(\{.*\})', current_message) # Meter values are logged as a dictionary
            temp_dic = json.loads(value.group(1))
            if temp_dic:
                match = int(temp_dic["connectorId"]) # Get the correct outletId
                meter_values[file_index][match].append(temp_dic["meterValue"]) 

        
        elif evse_values_message in current_message:

            value = re.findall(r'(\w+)\[(?:\w+|%)\]=\s*(\d+\.?\d*)', current_message) # The 'DCC' values are separated by pipes '|'. they follow a pattern : EVSE_xxxx[A] = yyy| .... 
            #   The information that comes before [A] is grabed by the first (\w+) group.
            #       The characters inside the square brackets correspond to the unit. The initial '?:' signals a non-capturing group.
            #   The final numerical information is the corresponding value
            dic_temp = {key:float(value) for key,value in value}

            evse_values[file_index][match].append([row[1]["@timestamp"], dic_temp])
        
        elif temperature_message in current_message:

            temperature_values[file_index][match].append(row[1]["@timestamp"])

        if error_message in current_message: # Events 
            if "true" in current_message: # Events have 'true' / 'false' keyword. I guess it marks wheter the error was solved or is logged in the first time
                code = 1
            else:
                code = -1

            if "Outlet_DC" in current_message: # In the first two cases, it is possible to assign an OutletID
                match = int(re.search(r"Outlet_DC(\d)_codes", current_message).group(1))
                if "true" in current_message: 
                    code = int(re.search(r"Errorcode:\s*(\d+)", current_message).group(1))
                string = "Outlet_DCX_code"
            elif "IMD_BC" in current_message:
                match = int(re.search(r"IMD_BC(\d)", current_message).group(1))
                string="IMD_BCX"

            # For events where we don't know the source outlet, save them in a default 'Dc=0' 
            elif "PowerConverter[" in current_message: 
                match = 0
                if "true" in current_message:
                    code = int(re.search(r"PowerConverter\[(\d+)\]", current_message).group(1)) # The numerical value here is a error code, and not the outlet ID
                string = "PowerConverter[Y]"
            else: # The other events are j
                match = 0
                string = re.search(r"'(\w+)'", current_message).group(1)
            error_values[file_index][match][string].append([row[1]["@timestamp"], code])



        elif send_command_message_1 in current_message and not send_command_message_2 in current_message:
            # print(current_message)        
            # Get information here
            pass

In [ ]:
# Get interesting messages from extended logs
extended_session_change_info = {}
extended_error_values = {}

extended_insulation_resistance = {}
extended_capacitance = {}

extended_meter_values = {}
extended_evse_values = {}

extended_status = {}
extended_alert = {}

extended_temperature_values = {}

extended_separated_transactions_resitances = {}
extended_separated_transactions_capacitances = {}
extended_separated_transactions_evse_values = {}
extended_separated_transactions_error_values = {}
extended_separated_transactions_status = {}
extended_separated_transactions_alert = {}

extended_dcx_dcx = []

# for file_index in [2]:
for file_index in range(5):
    info_dic = get_transactions_from_raw_log(os.path.join(DATA_DIR_EXTENDED_CLEAN, CLEAN_FILES_EXTENDED[file_index]))
    
    extended_session_change_info[file_index] = {i: info_dic["start_stops"][i] for i in info_dic["start_stops"]}
    extended_error_values[file_index] = {i: info_dic["error_values"][i] for i in info_dic["error_values"]}
    extended_insulation_resistance[file_index] = {i: info_dic["resistances"][i] for i in info_dic["resistances"]}
    extended_capacitance[file_index] = {i: info_dic["capacitances"][i] for i in info_dic["capacitances"]}
    extended_meter_values[file_index] = {i: info_dic["meter_values"][i] for i in info_dic["meter_values"]}
    extended_evse_values[file_index] = {i: info_dic["evse_values"][i] for i in info_dic["evse_values"]}

    extended_separated_transactions_resitances[file_index] = {1 : [], 2: []}
    extended_separated_transactions_capacitances[file_index] = {1 : [], 2: []}
    extended_separated_transactions_evse_values[file_index] = {i: [] for i in info_dic["start_stops"]}
    extended_separated_transactions_error_values[file_index] = {i: [] for i in info_dic["error_values"]}
    extended_separated_transactions_status[file_index] = []
    extended_separated_transactions_alert[file_index] = {i: [] for i in info_dic["alert"]}

    extended_dcx_dcx.append(info_dic["dcx_dcx"])
    
    for i in info_dic["start_stops"]:

        for transaction_index, transaction in enumerate(extended_session_change_info[file_index][i]):
            if transaction[0][1] == 1:
                temp = []
                start = transaction[0][0]
                end = transaction[-1][0]
                tolerance = pd.Timedelta('5m')
                if transaction_index < len(extended_session_change_info[file_index][i]) - 1:
                    tolerance =  np.min([(extended_session_change_info[file_index][i][transaction_index+1][0][0]-end)/2, tolerance])
                    tolerance = np.clip((extended_session_change_info[file_index][i][transaction_index+1][0][0]-end)/2 , pd.Timedelta('0m'), pd.Timedelta('5m'))

                extended_separated_transactions_resitances[file_index][i].append(assign_correct_values(extended_insulation_resistance[file_index][i], start, end, tolerance))
                extended_separated_transactions_capacitances[file_index][i].append(assign_correct_values(extended_capacitance[file_index][i], start, end, tolerance))

                evse_values_ = assign_correct_values(extended_evse_values[file_index][i], start, end, tolerance)
                # print(assign_correct_values(extended_insulation_resistance[file_index][i], start, end))
                
                evse_values_keys = set([key for timestamp, element in evse_values_ for key in element])
                evse_values_timestamps = np.array([ timestamp for timestamp, element in evse_values_ ])
                evse_values_measures = { measure: np.array([float(element[measure]) for timestamp, element in evse_values])
                                        for measure in evse_values_keys}
                extended_separated_transactions_evse_values[file_index][i].append([evse_values_timestamps, evse_values_measures])

                temp = assign_correct_values(extended_error_values[file_index][i], start, end, tolerance)
                temp = temp +  assign_correct_values(extended_error_values[file_index][0], start, end, tolerance)

                extended_separated_transactions_error_values[file_index][i].append(temp)

                # temp = assign_correct_values(info_dic["status"][i], start, end, tolerance)
                temp = []
                for j in range(5):
                    temp += assign_correct_values(info_dic["status"][j], start,end,tolerance)
                # temp = temp + assign_correct_values(info_dic["status"][0], start, end, tolerance)
                extended_separated_transactions_status[file_index].append(temp)
                extended_separated_transactions_alert[file_index][i].append(assign_correct_values(info_dic["alert"][i], start, end, tolerance))


In [ ]:

# Check wheter a particular message is popular:

specific_message = "DPA.PwrRealign"

times_per_datafile = []
for i in range(4):
    data = pd.read_csv(os.path.join(DATA_DIR_CLEAN, CLEAN_FILES[i]))
    data["@timestamp"] = pd.to_datetime(data["@timestamp"])
    # data["@timestamp"] = data["@timestamp"].dt.tz_localize('UTC').dt.tz_convert('UTC+05:30')
    data = data.sort_values(by="@timestamp", ascending=True)
    times = []
    tot_lines = 0
    for row in data.iterrows():
        if specific_message in row[1]["@message"]: 
            tot_lines += 1
            times.append((row[1]["@timestamp"]-times_of_explosion[i]).total_seconds()/60)
            # times.append(times_of_explosion[i]-row[1]["@timestamp"])
            # print(pd.Timestamp(times_of_explosion[i])-pd.Timestamp(row[1]["@timestamp"])+pd.Timedelta('0m'))
    
    times_per_datafile.append([times, tot_lines])

In [ ]:
file_name = os.path.join(DATA_DIR_BIG_CLEAN, CLEAN_FILES_BIG[0])
info_dic = get_transactions_from_raw_log(file_name)
transactions_start_stops, transactions_resistances, error_values = info_dic["start_stops"], info_dic["resistances"], info_dic["error_values"]

In [ ]:
separated_transactions_resitances = {1: [], 2: [], 3: [], 4:[]}
separated_transactions_capacitances = {1: [], 2: [], 3: [], 4:[]}
separated_transactions_evse_values = {1: [], 2: [], 3: [], 4:[]}
tolerance = pd.Timedelta('5m')
transaction_dictionary = {1: [], 2: [], 3: [], 4:[]}

safety_relay = []
ev_dropped   = []

main_fan = []
dcx_dcx = []
charging_phase = []
pilot_state = []

num = 0
num2 = 0
for name in CLEAN_FILES_BIG:

    file_name = os.path.join(DATA_DIR_BIG_CLEAN, name)
    print(file_name)
    temp1 = []
    temp2 = []

    
    info_dic = get_transactions_from_raw_log(file_name) 
    transactions_start_stops, transactions_resistances, transaction_evse_values, transactions_capacitances  = info_dic["start_stops"], info_dic["resistances"], info_dic["evse_values"], info_dic["capacitances"]

    status_not = info_dic["status"]
    for status_list_index, status_list in enumerate(status_not):
            for status in status_not[status_list]:
                if status[1]["status"] == "Faulted":
                    if "vendorErrorCode" in status[1].keys():
                        if "SafetyRelay" in status[1]["vendorErrorCode"]:
                            # print(status)
                            num += 1
                            temp1.append(status)
                            pass

    alert = info_dic["alert"]
    for alert_list_index, alert_list in enumerate(alert):
            for al in alert[alert_list]:
                if "ERR_EV_DROPPED, vehicle ended with unexpected pilot state B" in  al[1]["payload"]["msg"]:
                    temp2.append(al)
                    # print(al)
                    num2 += 1
                    pass

    safety_relay.append(sorted(temp1, key=lambda x: x[0]))
    ev_dropped.append(sorted(temp2, key=lambda x: x[0]))

    main_fan.append(info_dic["main_fan"][0])
    
    dcx_dcx_dic = info_dic["dcx_dcx"]
    temp = []
    for dcx_dcx_list in dcx_dcx_dic:
        for dcx_dcx_element in dcx_dcx_dic[dcx_dcx_list]:
            temp.append(dcx_dcx_element)

    dcx_dcx.append(temp)

    charging_phase_dic = info_dic["charging_phase"]
    temp = []
    for charging_phase_list in charging_phase_dic:
        for charging_phase_element in charging_phase_dic[charging_phase_list]:
            temp.append(charging_phase_element)
    charging_phase.append(temp)

    pilot_state_dic = info_dic["pilot_state"]
    temp = []
    for pilot_state_list in pilot_state_dic:
        for pilot_state_element in pilot_state_dic[pilot_state_list]:
            temp.append(pilot_state_element)

    pilot_state.append(temp)

    for i in range(1,5):
        index_transaction = 0
        for transaction in transactions_start_stops[i]:
            if transaction[0][1] == 1:
                temp = []
                start = transaction[0][0]
                end   = transaction[-1][0]

                # index_resistance = bisect_left(transactions_resistances[i], start-tolerance, key= lambda x: x[0])

                # while index_resistance < len(transactions_resistances[i]) and transactions_resistances[i][index_resistance][0] <= end+tolerance:
                #     temp.append(transactions_resistances[i][index_resistance])
                #     index_resistance += 1
                # temp.append(transactions_resistances[i][index_resistance])
                separated_transactions_resitances[i].append(assign_correct_values(transactions_resistances[i],start, end))
                separated_transactions_capacitances[i].append(assign_correct_values(transactions_capacitances[i],start, end))
                evse_values_ = assign_correct_values(transaction_evse_values[i], start, end)
                
                evse_values_keys = set([key for timestamp, element in evse_values_ for key in element])
                evse_values_timestamps = np.array([ timestamp for timestamp, element in evse_values_ ])
                evse_values_measures = { measure: np.array([float(element[measure]) for timestamp, element in evse_values_])
                                        for measure in evse_values_keys}
                separated_transactions_evse_values[i].append([evse_values_timestamps, evse_values_measures])

                transaction_dictionary[i].append([file_name,index_transaction])
                index_transaction += 1

print(num, num2)


# Graphs

## Particular Message

In [ ]:
fig, axs = plot.subplots(1,2, figsize=(12,5), sharey=True)

ax = axs[0]
SHOW = 0
prepare_subplots(axs)

markers = ["o", "s", "D", "X"]
for i in range(4):
    axs[0].scatter(times_per_datafile[i][0], [i]*len(times_per_datafile[i][0]), marker=markers[i], edgecolor="k", label=times_per_datafile[i][1])
    axs[1].scatter(times_per_datafile[i][0], [i]*len(times_per_datafile[i][0]), marker=markers[i], edgecolor="k", label=times_per_datafile[i][1])

for a in axs:
    leg = a.legend(title="# Total messages", loc="lower left")
    a.add_artist(leg)

# axs[0].set_xlim(-120,10)
axs[1].set_xlim(-2,2)

lines = [
    mlines.Line2D([],[],color="r", label="Explosion"),
    mlines.Line2D([],[],color="k", marker=".", linestyle="none", label="Timestamp")
]
for a in axs:
    a.set_xlabel(r"$t_{message}-\hat{t}_{explosion}$ (min)", size=15)
    a.axvline(0, color="r", label="Explosion",zorder=0)
    a.legend(handles = lines, loc="upper left")

fig.suptitle("Time stamps of specific message", size=15)

if not SHOW:
    plot.close()




## Resistance & Capacitance

In [ ]:
fig, axs = plot.subplots(1,2, figsize=(14,5))
# fig, axs = plot.subplots(2,1, figsize=(6,12))

fig2, axs2 = plot.subplots(1,2, figsize=(14,5))
# fig2, axs2 = plot.subplots(2,1, figsize=(6,12))

SHOW = 1
figs = [fig, fig2]
axs = [*axs,*axs2]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")
# colors = color_dic["default"]
# prepare_subplots(axs, key="default")


colors = color_dic["default"]
linestyles = { 1: "-", 2:"-" }

# for i in range(4):
alphas_1 = [0.3,1,1,0.3]
alphas_2 = [1,0.1,0.1,1]
marker_dic = {"Outlet_DCX_code":"X", "IMD_BCX":"s", "PowerConverter[X]":"^", "other": "d"}
for i in [3,0,2,1]:
    for dc in range(1,3,1):
        if (i == 1 and dc == 2) or (i==0 and dc==2):
            continue
        
        xs = [(val[0]-times_of_explosion[i]).total_seconds()/60 for val in insulation_resistance[i][dc]]
        ys = [val[1] for val in insulation_resistance[i][dc]]
        lines, = axs[0].plot(xs, ys, marker=".", color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_1[i])
        axs[1].plot(xs, ys, marker=".", color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_2[i])


        xs = [(val[0]-times_of_explosion[i]).total_seconds()/60 for val in capacitance[i][dc]]
        ys = [val[1] for val in capacitance[i][dc]]
        lines, = axs2[0].plot(xs, ys, marker=".", color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_1[i])
        axs2[1].plot(xs, ys, marker=".",color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_2[i])
    # xs = [val[0] for val in insulation_resistance[i]]
    # axs2[1].scatter(xs[-2:], [0]*len(xs[-2:]), marker="|", color=lines.get_color())

        xs_start = [(val-times_of_explosion[i]).total_seconds()/60 for val in session_change_info[i]["start_charging"][dc] ]
        xs_end = [(val-times_of_explosion[i]).total_seconds()/60 for val in session_change_info[i]["stop_charging"][dc] ]

        
        
        for a in axs:
            pass
            # a.scatter(xs_start, [0]*len(xs_start), marker="s", color=colors[i])
            # a.scatter(xs_end, [0]*len(xs_end), marker="x", color=colors[i])

        # axs2[0].scatter(xs_start, [0]*len(xs_start), marker="s", color=colors[i])
        # axs2[0].scatter(xs_end, [0]*len(xs_end), marker="x", color=colors[i])

lines = [
    mlines.Line2D([],[],color="r", label="Separation", linestyle=":"),
    # mlines.Line2D([],[],color="k", marker=".", linestyle="none", label="Data Point")
    # mlines.Line2D([],[],color="k", marker="*", linestyle="none", label="Error")
]

Ids = [
    mpatches.Patch(facecolor=colors[0], label="0", edgecolor="k"),
    mpatches.Patch(facecolor=colors[1], label="1", edgecolor="k"),
    mpatches.Patch(facecolor=colors[2], label="3", edgecolor="k"),
    mpatches.Patch(facecolor=colors[3], label="4", edgecolor="k"),
]

for a in [*axs, *axs2]:
    a.set_xlabel(r"$t_{message}-\hat{t}_{separation}$ (min)", size=18)
    a.axvline(0, color="r", zorder=0, linestyle=":")

for a in axs[:2]:
    a.set_ylabel("Insulation resistance (kOhm)", size=18)
    a.set_ylim(0,2250)
    leg = a.legend(handles = lines, loc="lower left")
    a.add_artist(leg)
    leg = a.legend(handles = Ids, loc="lower center", ncol=2, title="Transaction ID")

for a in axs2:
    a.set_ylim(-0.1, 1.3)
    a.set_ylabel("Leakage Capacitance (microF)", size=18)
    leg = a.legend(handles = lines, loc="lower right")
    a.add_artist(leg)
    leg = a.legend(handles = Ids, loc="lower center", ncol=2, title="Transaction ID")

fig.suptitle("Evolution of Insulation Resistance", fontsize=18)
# fig2.suptitle("Evolution of Capacitance", fontsize=15)

axs[1].set_xlim(-40,5)
axs2[1].set_xlim(-40,5)
if not SHOW:
    for a in axs:
        plot.close()


## Investigate Events

In [ ]:
fig, axs = plot.subplots(2,1,figsize=(14,10))


SHOW = 0
prepare_subplots(axs)


colors = color_dic["default"]
linestyles = { 1: "-", 2:"-" }

# for i in range(4):
alphas_1 = [0.3,1,1,0.3]
alphas_2 = [1,0.1,0.1,1]
marker_dic = {
    "Outlet_DCX_code": "X",
    "IMD_BCX": "s",
    "PowerConverter[Y]": "^",
    "Other": "d",
}
for i in [3,0,2,1]:

    for dc in range(1,3,1):
        if (i == 1 and dc == 2) or (i==0 and dc==2):
            continue
        xs = [(val[0]-times_of_explosion[i]).total_seconds()/60 for val in insulation_resistance[i][dc]]
        ys = [val[1] for val in insulation_resistance[i][dc]]
        axs[0].plot(xs, ys, marker=".", color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_1[i])
        axs[1].plot(xs, ys, marker=".", color=colors[i], linestyle=linestyles[dc],zorder=4,markeredgecolor="k", alpha=alphas_2[i])

        

        for a,alph in zip(axs, [alphas_1, alphas_2]) : # A somewhat convoluted way of marking the events automatically
            for error_key in list(marker_dic.keys()) + list(error_values[i][0].keys()):
                dc_c = dc # Get a temporary outlet ID
                marker_error_key = error_key
                if error_key not in marker_dic.keys() or error_key == "PowerConverter[Y]":
                    dc_c = 0 # Assign the correct Id. If the error did not have outlet information it was stored in the neutral 'dc=0'.
                    if error_key != "PowerConverter[Y]":
                        marker_error_key = "Other" 
                for element in error_values[i][dc_c][error_key]:
                    if len(ys) == 0:
                        continue
                    time_now = (element[0]-times_of_explosion[i]).total_seconds()/60
                    idx = bisect_left(xs, time_now)
                    if idx > 0:
                        idx-= 1
                    a.scatter(time_now, ys[idx], marker=marker_dic[marker_error_key], color=colors[i], zorder=5, edgecolors="k", s=50, alpha=alph[i])

lines = [
    mlines.Line2D([],[],color="r", label="Explosion"),
    mlines.Line2D([],[],color="k", marker=".", linestyle="none", label="Data Point")
]

Ids = [
    mpatches.Patch(facecolor=colors[0], label="0", edgecolor="k"),
    mpatches.Patch(facecolor=colors[1], label="1", edgecolor="k"),
    mpatches.Patch(facecolor=colors[2], label="3", edgecolor="k"),
    mpatches.Patch(facecolor=colors[3], label="4", edgecolor="k"),
]

markers = [ mlines.Line2D([],[],color="k", marker=marker_dic[key], label=key,linestyle="none" ) for key in ["Outlet_DCX_code","IMD_BCX", "PowerConverter[Y]", "Other"]]

for a in axs:
    a.set_xlabel(r"$t_{message}-\hat{t}_{explosion}$ (min)", size=15)
    a.axvline(0, color="r", zorder=0)

for a in axs:
    a.set_ylabel("Insulation resistance (kOhm)", size=15)
    a.set_ylim(0,2250)
    leg = a.legend(handles = lines, loc="lower left")
    a.add_artist(leg)
    leg = a.legend(handles = Ids, loc="lower center", ncol=2, title="Transaction ID")
    a.add_artist(leg)
    leg = a.legend(handles = markers, loc="upper center", ncol=1, title="Marker ID")


fig.suptitle("Evolution of Insulation Resistance", fontsize=15)
axs[0].set_xlim(-120,5)
axs[1].set_xlim(-40,5)

if not SHOW:
    for a in axs:
        plot.close()




## Charger Information

In [ ]:
keys = [dic["measurand"] for dic in meter_values[0][1][0][0]['sampledValue']]
keys_2 = list(evse_values[0][1][0][1].keys())

xs = [ {dc: np.array([ (pd.Timestamp(element[0]['timestamp'][:-1])-times_of_explosion[i]).total_seconds()/60 for element in meter_values[i][dc]]) for dc in range(1,3)} for i in range(4) ]

values = [ {dc: {key: np.array([float(dic['value']) for element in meter_values[i][dc] for dic in element[0]['sampledValue'] if dic['measurand'] == key]) for key in keys}
            for dc in range(1,3)}
          for i in range(4)]

xs_2  = [ {dc: np.array([ (element[0]-times_of_explosion[i]).total_seconds()/60 for element in evse_values[i][dc]]) for dc in range(1,3)} for i in range(4) ]

values_2 = [ {dc: {key: np.array([float(element[1][key]) for element in evse_values[i][dc]]) for key in keys_2}
            for dc in range(1,3)}
          for i in range(4)]

print(keys)
print(keys_2)


In [ ]:
SHOW = 0
fig, axs = plot.subplots(4,1,sharex=True, figsize=(8,12))

figs = [fig]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

fig2, ax2= plot.subplots(1,2, sharex=True, sharey=True)
fig3, ax3= plot.subplots()
fig4, ax4 = plot.subplots(1,3, figsize=(19,5))
prepare_subplots([*ax2, ax3, *ax4])
bins = np.arange(0,120,1)

ax5 = ax4[1]
ax6 = ax4[2]
ax4 = ax4[0]

linestyles = { 1: "-", 2:"--" }

# for i in [0,1,2,3]:
colors = color_dic['default']

show = [0,1,2,3]


for i in show:

    for dc in [1,2]:

        if (i == 1 and dc == 2) or (i==0 and dc==2):
            continue
        wrong_index = (np.where(values_2[i][dc]["EVSE_NowPwr"] == -1e2)[0])
        FLAG = False
        if len(wrong_index):
            print(wrong_index, dc)
            for index in wrong_index:
                values_2["EVSE_NowPwr"][index] = 0
            FLAG = True

        if len(xs[i][dc]) == len(values[i][dc]['Voltage']):
            axs[0].plot(xs[i][dc], values[i][dc]['Voltage'], color=colors[i], marker=".", markersize=2,linestyle=linestyles[dc])
        if len(xs_2[i][dc]) == len(values_2[i][dc]['EVSE_NowVol']):
            axs[0].plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowVol'], color=colors[i], linestyle=":")

        if len(xs[i][dc]) == len(values[i][dc]['Current.Offered']):
            # axs[1].plot(xs, values['Current.Offered'], color=colors[i], markersize=2, linestyle=linestyles[dc])
            axs[1].plot(xs[i][dc], values[i][dc]['Current.Import'], color=colors[i], markersize=2, linestyle=(0,(3,0)))
        if len(xs_2[i][dc]) == len(values_2[i][dc]['EVSE_NowCur']):
            axs[1].plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowCur'], color=colors[i], linestyle=":")
            

        if len(xs[i][dc]) == len(values[i][dc]['Power.Offered']):
            axs[2].plot(xs[i][dc], values[i][dc]['Power.Offered'], color=colors[i], marker=".", markersize=2, linestyle=linestyles[dc])
        if len(xs_2[i][dc]) == len(values_2[i][dc]['EVSE_NowPwr']):
            axs[2].plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowPwr']*1e3, color=colors[i], linestyle=":")

        if len(xs[i][dc]) == len(values[i][dc]['SoC']):
            axs[3].plot(xs[i][dc], values[i][dc]['SoC'], color=colors[i], marker=".", markersize=2, linestyle=linestyles[dc])
        if len(xs_2[i][dc]) == len(values_2[i][dc]['EV_SOC']):
            axs[3].plot(xs_2[i][dc], values_2[i][dc]['EV_SOC'], color=colors[i], linestyle=":")
        
        diffs = (xs[i][dc][1:]-xs[i][dc][:-1])*60 # seconds
        diffs_2 = (xs_2[i][dc][1:]-xs_2[i][dc][:-1])*60 # seconds

        ax2[0].hist(diffs_2, bins=bins, color=colors[i], hatch=linestyles[dc])
        ax2[1].hist(diffs, bins=bins, color=colors[i], hatch=linestyles[dc])
        ax3.hist(diffs, bins=bins, color=colors[i], label=CLEAN_FILES[i])

        # print(len(diffs))
        # print(i, dc, diffs)

        first_volt = np.where(values_2[i][dc]['EVSE_NowVol']>600)[0]
        if len(first_volt):
            first_volt = first_volt[0]
            last_volt = np.argmax(values_2[i][dc]['EVSE_NowVol'])

            print(values_2[i][dc]['EVSE_NowVol'][first_volt], values_2[i][dc]["EVSE_NowVol"][last_volt])
            print((-values_2[i][dc]['EVSE_NowVol'][first_volt] + values_2[i][dc]["EVSE_NowVol"][last_volt])/values_2[i][dc]["EVSE_NowVol"][first_volt]*100)
for i in show:
    for dc in [1,2]:
        xs_start = [(val-times_of_explosion[i]).total_seconds()/60 for val in session_change_info[i]["start_charging"][dc] ]
        xs_end = [(val-times_of_explosion[i]).total_seconds()/60 for val in session_change_info[i]["stop_charging"][dc] ]
        for a in axs:
            ymin, ymax = a.get_ylim()
            a.scatter(xs_start, [0.5*(ymin+ymax)]*len(xs_start), marker="s", color=colors[i])
            a.scatter(xs_end, [0.5*(ymin+ymax)]*len(xs_end), marker="x", color=colors[i])
    


i, dc = 2, 2
ax4.plot(xs[i][dc], values[i][dc]["Current.Import"], color=colors[i], linestyle="-", label="Current.Import")
ax4.plot(xs[i][dc], values[i][dc]["Current.Offered"], color=colors[i], linestyle="--", label="Current.Offered")
leg = ax4.legend(title="Meter Values:", loc="upper left")
ax4.add_artist(leg)
line_miss, = ax4.plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowCur'], color="#636363", linestyle="--", label="EVSE_NowCur")
ax4.legend(title="DCC Values", handles=[line_miss], loc="upper right")

ax5.plot(xs[i][dc], values[i][dc]["Power.Active.Import"]/1e3, color=colors[i], linestyle="-", label="Power.Active.Import")
ax5.plot(xs[i][dc], values[i][dc]["Power.Offered"]/1e3, color=colors[i], linestyle="--", label="Power.Offered")

leg = ax5.legend(title="Meter Values:", loc="upper left")
ax5.add_artist(leg)
line_miss, = ax5.plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowPwr'], color="#636363", linestyle="--", label="EVSE_NowPwr")
ax5.legend(title="DCC Values", handles=[line_miss], loc="upper right")

ax6.plot(xs_2[i][dc], values_2[i][dc]['EVSE_NowVol'], color="#636363", linestyle="--", label="EVSE_NowVol")
ax6.legend(title="DCC Values", loc="upper right")





axs[0].set_title("Voltage",size=18)
axs[0].set_ylabel("V",size=18)

ax6.set_ylabel("Voltage (V)",size=18)

axs[1].set_title("Current.Offered",size=18)
axs[1].set_ylabel("A",size=18)

# ax4.set_title("Current Differences",size=18)
ax4.set_ylabel("Current (A)",size=18)

axs[2].set_title("Power.Offered",size=18)
axs[2].set_ylabel("W",size=18)

# ax5.set_title("Power Differences",size=18)
ax5.set_ylabel("Power (kW)",size=18)


axs[3].set_title("SoC", size=18)
axs[3].set_ylabel("%", size=18)

axs[-1].set_xlabel(r"$t_{message}-\hat{t}_{separation}$ (min)", size=18)
ax4.set_xlabel(r"$t_{message}-\hat{t}_{separation}$ (min)", size=18)
ax5.set_xlabel(r"$t_{message}-\hat{t}_{separation}$ (min)", size=18)

ax4.set_ylim(-10,320)
ax5.set_ylim(-10,320)
ax6.set_ylim(-10,820)

fig4.suptitle(f"Differences between 'Meter' and 'DCC' values using: {CLEAN_FILES[i].split("_")[0]+".csv"}", fontsize=18)

ax2[0].set_ylabel("# Instances")
ax2[0].set_xlabel("Diff in logs (seconds)")

ax3.set_ylabel("# Instances", fontsize=18)
ax3.set_title("Time lag between subsequent log messages \'Meter Values\'", fontsize=18)
ax3.set_xlabel(r"$t_{i+1} - t_{i}$", fontsize=18)
ax3.legend()

if not SHOW:
    for a in [*axs, ax2, ax3]:
        plot.close()



In [ ]:
SHOW = 1

fig, ax = plot.subplots(1,2, figsize=(14,5))
fig2, ax2 = plot.subplots(1,2, figsize=(14,5))

axs = [*ax, *ax2]
figs = [fig, fig2]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

colors = color_dic["default"]
alphas = [[0.4,1,1,0.4], [1,0.2,0.2,1]]
for file_index in [3,0,2,1]:
    for dc in range(1,3,1):
        if (file_index == 1 and dc == 2) or (file_index==0 and dc==2) or (file_index==2 and dc==1) or (file_index==3 and dc==2): continue

        

        for index_a, a in enumerate(ax):
            a.plot(xs_2[file_index][dc], values_2[file_index][dc]["EVSE_NowVol"], color=colors[file_index], alpha=alphas[index_a][file_index])
        for index_a, a in enumerate(ax2):
            a.plot(xs_2[file_index][dc], values_2[file_index][dc]["EVSE_NowCur"], color=colors[file_index], alpha=alphas[index_a][file_index])

Ids = [
    mpatches.Patch(facecolor=colors[0], label="0", edgecolor="k"),
    mpatches.Patch(facecolor=colors[1], label="1", edgecolor="k"),
    mpatches.Patch(facecolor=colors[2], label="3", edgecolor="k"),
    mpatches.Patch(facecolor=colors[3], label="4", edgecolor="k"),
]

lines = [
    mlines.Line2D([],[],color="r", label="Separation", linestyle=":"),
    # mlines.Line2D([],[],color="k", marker=".", linestyle="none", label="Data Point")
    # mlines.Line2D([],[],color="k", marker="*", linestyle="none", label="Error")
]

for a in axs:
    a.set_xlabel(r"$t_{message}-\hat{t}_{separation}$ (min)", size=15)
    a.axvline(0, color="r", zorder=-1, linestyle=":")
    leg = a.legend(handles = Ids, loc="lower center", ncol=2, title="Transaction ID")
    a.add_artist(leg)
    leg = a.legend(handles = lines, loc="lower right")

for a in ax:
    a.set_ylim(-20,750)
    a.set_ylabel("Voltage (V)", fontsize=15)

for a in ax2:
    a.set_ylim(-10,210)
    a.set_ylabel("Current (A)", fontsize=15)

ax[0].set_xlim(-120,5)
ax2[0].set_xlim(-120,5)
    
ax[1].set_xlim(-40,5)
ax2[1].set_xlim(-40,5)


if not SHOW:
    for a in axs:
        plot.close()


## Compare Resistances

In [ ]:
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()

axs = [ax, ax2, ax3]
figs = [fig, fig2, fig3]
SHOW = 0
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

# for dc in range(1,5):
color_lines = colors[1]
for dc in [1,2]:
    for index_transaction, transaction in enumerate(separated_transactions_resitances[dc]):
    
        # if index_transaction != 3:
            # continue
        # start,stop = transactions_start_stops[dc][index_transaction]
        # start = start[0]
        # stop = stop[0]
        xs = np.array([(element[0]-transaction[0][0]).total_seconds() for element in transaction])
        ys = np.array([element[1] for element in transaction])
        # total = (stop-start).total_seconds()
        if len(xs):
            total = (xs[-1])


            alpha = 0.1
            ax.plot(xs[1:-1]/total, ys[1:-1],zorder=1, color=color_lines, linestyle="-", markersize=4, alpha=0.1, linewidth=1.5)
            # ax.scatter(xs[0]/total, ys[0],marker="s", zorder=4, color="g", alpha=0.1)
            # ax.scatter(xs[-1]/total, ys[-1],marker="s", zorder=4, color="r", alpha=0.1)

            if np.any((0) | ((ys<550) & (ys !=1))):
                # print(index_transaction)
                pass
        else:
            # print("The transaction ", index_transaction, " has no resistance info")
            pass

# ax.set_xlim(-0.0001,0.005)
inis = [transaction[0][1]  for i in range(1,5) for transaction in separated_transactions_resitances[i] if len(transaction)]
finis = [transaction[0][1] for i in range(1,5) for transaction in separated_transactions_resitances[1] if len(transaction)]
ax2.hist(inis)
ax3.hist(finis)

# Exploding cases
colors = color_dic["default"]
for i in range(4):
    for dc in range(1,3,1):
        if (i == 1 and dc == 2) or (i==0 and dc==2) or (i==2 and dc==1) or (i==3 and dc==2):
            continue
        array = insulation_resistance[i][dc]
        xs = np.array([(val[0]-array[0][0]).total_seconds() for val in array])
        ys = np.array([val[1] for val in array])

        ax.plot(xs/xs[-1], ys, color=colors[i],  linestyle="-", linewidth=1.5)



ax.set_xlabel(r"$\frac{t-t_{min}}{t_{max}-t_{min}}$", fontsize=18)
ax.set_xlabel(r"$\frac{t}{D}$", fontsize=18)
ax.set_ylabel(r"Insulation resistance (kOhm)", fontsize=18)

ax2.set_xlabel(r"Insulation resistance (kOhm)", fontsize=18)
ax2.set_ylabel(r"# Instances", fontsize=18)
ax2.set_title(r"Initial Resitance Value", fontsize=18)
ax3.set_xlabel(r"Insulation resistance (kOhm)", fontsize=18)
ax3.set_ylabel(r"# Instances", fontsize=18)
ax3.set_title(r"Final Resitance Value", fontsize=18)

# ax.set_ylim(0,3)
ax.set_title("Insulation Resistances reported by 23 charger-dataset", fontsize=18, pad=15)

Ids = [
    mpatches.Patch(facecolor=colors[0], label="0", edgecolor="k"),
    mpatches.Patch(facecolor=colors[1], label="1", edgecolor="k"),
    mpatches.Patch(facecolor=colors[2], label="3", edgecolor="k"),
    mpatches.Patch(facecolor=colors[3], label="4", edgecolor="k"),
    mpatches.Patch(facecolor=color_lines, label="Other", edgecolor="k"),
]

leg = ax.legend(handles = Ids, loc="lower center", ncol=5, title="Transaction ID")

# ax.set_ylim(-0.2,2)

for f in figs:
    # f.tight_layout()
    pass
if not SHOW:
    for a in axs:
        plot.close()


In [ ]:
fig, ax = plot.subplots()
fig4, ax4 = plot.subplots()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()

SHOW = 1

axs = [ax, ax2, ax3, ax4]
figs = [fig, fig2, fig3, fig4]

colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

color_lines     =colors[1]
color_lines_bad = "#A72666" #colors[-5]

temp = []
temp2 = []
sum_dys = []
full_transaction_index = 0
for dc in [1,2]:
    for index_transaction, transaction in enumerate(separated_transactions_resitances[dc]):
        # if index_transaction != 160:
        #     continue
        # if index_transaction in temp:
        #     # continue
        #     pass
        xs = np.array([(element[0]-transaction[0][0]).total_seconds() for element in transaction])
        ys = np.array([element[1] for element in transaction])
        if len(xs)>1:
            total = xs[-1]

            new_ys = ys[1:-1] if len(xs) > 2 else ys
            # if np.std(new_ys) > 80:
            #     continue

            dys = np.abs(ys[1:]-ys[:-1])/(xs[1:]-xs[:-1])
            color=color_lines
            zorder=1
            alpha=0.3
            # if not np.sum(dys>20)>2:
            if np.sum(dys) > 100:
                color=color_lines_bad
                zorder=0
                alpha=0.3
                ax4.plot(xs[1:-1]/total, ys[1:-1], color=color, alpha=alpha, linewidth=1.5,zorder=zorder)

                temp.append(full_transaction_index)
            ax.plot(xs[1:-1]/total, ys[1:-1], color=color, alpha=alpha, linewidth=1.5,zorder=zorder)
            # ax2.scatter(index_transaction, np.std(new_ys), color="k")
            ax2.plot(xs[:-1]/xs[-1], dys,color="k", alpha=0.3,linewidth=1.5)
            ax3.scatter(index_transaction, np.max(dys), color="k")
        else:
            temp2.append(full_transaction_index)
        
        full_transaction_index += 1
            

print(len(temp), len(temp2), len(separated_transactions_resitances[1])+len(separated_transactions_resitances[2]))
# Exploding cases
colors = color_dic["default"]
for i in range(4):
    for dc in range(1,3,1):
        if (i == 1 and dc == 2) or (i==0 and dc==2) or (i==2 and dc==1) or (i==3 and dc==2):
            continue
        array = insulation_resistance[i][dc]
        xs = np.array([(val[0]-array[0][0]).total_seconds() for val in array])
        ys = np.array([val[1] for val in array])

        ax4.plot(xs/xs[-1], ys, color=colors[i],  linestyle="-", linewidth=1.5)

ax2.set_ylim(-1e1,1e2)
# ax.set_ylim(0,2100)

ax3.set_ylim(-10,500)

ax.set_xlabel(r"$\frac{t}{D}$", fontsize=18)
ax.set_ylabel(r"Insulation resistance (kOhm)", fontsize=18)
ax4.set_xlabel(r"$\frac{t}{D}$", fontsize=18)
ax4.set_ylabel(r"Insulation resistance (kOhm)", fontsize=18)

Ids = [
    mpatches.Patch(facecolor=colors[0], label="0", edgecolor="k"),
    mpatches.Patch(facecolor=colors[1], label="1", edgecolor="k"),
    mpatches.Patch(facecolor=colors[2], label="3", edgecolor="k"),
    mpatches.Patch(facecolor=colors[3], label="4", edgecolor="k"),
    mpatches.Patch(facecolor=color_lines_bad, label="Err", edgecolor="k"),
]

leg = ax4.legend(handles = Ids, loc="lower center", ncol=5, title="Transaction ID")

Ids = [
    mpatches.Patch(facecolor=color_lines, label="Normal", edgecolor="k"),
    mpatches.Patch(facecolor=color_lines_bad, label="Erratic", edgecolor="k"),
]
leg = ax.legend(handles = Ids, loc="lower center", ncol=5, title="Behaviour:")
ax.set_title("Insulation Resistances reported by 23 charger-dataset", fontsize=18, pad=15)
ax4.set_title("Comparisson between erratic behaviour and cable-explosion", fontsize=18, pad=15)

if not SHOW:
    for a in axs:
        plot.close()


## Metering Values

In [ ]:
fig, ax = plot.subplots()

SHOW = 1
axs = [ax]
prepare_subplots(axs)
dc = 2

alpha_filter = 1.
for dc in [2]:
    for index_transaction, transaction in enumerate(separated_transactions_evse_values[dc]):
        xs = np.array([(element-transaction[0][0]).total_seconds() for element in transaction[0]])
        ys = transaction[1]
    
        if len(ys):

            test = ys['EVSE_NowVol'].copy()
            for i in range(1,len(test)):
                test[i] = alpha_filter*test[i] + (1-alpha_filter)*test[i-1]
            ax.plot((xs)/(xs[-1]), test, color="k", alpha=0.1)

# Exploding cases
colors = color_dic["default"]
for i in range(4):
    for dc in range(1,3,1):
        if (i == 1 and dc == 2) or (i==0 and dc==2) or (i==2 and dc==1) or (i==3 and dc==2):
            continue
        
        keys = set([key for timestamp, element in evse_values[i][dc] for key in element])
    
        xs = np.array([(element[0]-evse_values[i][dc][0][0]).total_seconds() for element in evse_values[i][dc]])
        ys = { measure: np.array([float(element[measure]) for timestamp, element in evse_values[i][dc]])
                                for measure in keys}
        test = ys['EVSE_NowCur'].copy()
        if np.all(test == 0):
            continue
        for j in range(1,len(test)):
            test[j] = alpha_filter*test[j] + (1-alpha_filter)*test[j-1]

        # ax.plot(xs/xs[-1], test, color=colors[i], linestyle="--")

ax.set_xlabel(r"$\frac{t-t_{min}}{t_{max}-t_{min}}$", fontsize=18)
ax.set_ylabel(r"Current (A)", fontsize=15)

if not SHOW:
    plot.close()


### Compare with historical values

In [ ]:
fig, ax = plot.subplots()
fig3, ax3 = plot.subplots()
fig2, ax2 = plot.subplots()

axs = [ax,ax2,ax3]
prepare_subplots(axs)

axs2 = [ax,ax3]

alpha_filter=1
colors = color_dic["default"]
# for file_index in range(2):
for file_index in [2]:
    print(CLEAN_FILES_EXTENDED[file_index])

    file_index_clean = 2
    dc_clean = 2
    for dc in [1,2]:

        if (file_index == 1 and dc == 2) or (file_index==0 and dc==2) or (file_index == 2 and dc == 1) or (file_index == 3 and dc==2):
            continue
        start_exploding = insulation_resistance[file_index_clean][dc_clean][0][0]
        print(start_exploding)
        print(insulation_resistance[file_index_clean][dc_clean][-1][0])
        print(times_of_explosion[file_index_clean])
        
        for index_resistance, resistances_ in enumerate(extended_separated_transactions_resitances[file_index][dc]):
            # resistances = sorted(resistances, key=lambda x: x[0])
            for ax_index, resistances in enumerate([resistances_, extended_separated_transactions_capacitances[file_index][dc][index_resistance]]):
                xs = np.array([(val[0]) for val in resistances])
                if len(xs)>1:
                    alpha=0.1
                    # if np.abs((start_exploding-xs[0].tz_localize(None)).total_seconds()) < 10:
                    if (start_exploding-pd.Timedelta('10s')<xs[0].tz_localize(None)):
                        alpha= 1
                        errors = extended_separated_transactions_error_values[file_index][dc][index_resistance]
                        # continue
                
                    xs = np.array([(x-xs[0]).total_seconds() for x in xs])
                    ys = np.array([val[1] for val in resistances])
                    axs2[ax_index].plot(xs/xs[-1], ys, alpha=alpha, color=colors[file_index], marker=".")

            
        

        xs = np.array([(val[0]) for val in insulation_resistance[file_index_clean][dc_clean]])
        xs = np.array([(x-xs[0]).total_seconds() for x in xs])
        ys = np.array([val[1] for val in insulation_resistance[file_index_clean][dc_clean]])
        # ax.plot(xs/xs[-1], ys, alpha=alpha, color=colors[file_index_clean], linestyle="--")
        
        for evse_values in extended_separated_transactions_evse_values[file_index][dc]:
            xs = np.array([timestamp for timestamp in evse_values[0]])
            if len(xs):
                alpha=0.1
                if np.abs((start_exploding-xs[0].tz_localize(None)).total_seconds()) < 90:
                    alpha= 1
                xs = np.array([(x-xs[0]).total_seconds() for x in xs])
                ys = evse_values[1]["EVSE_NowCur"].copy()
                if np.all(ys==0):
                    continue

                for j in range(1,len(ys)):
                    ys[j] = alpha_filter*ys[j] + (1-alpha_filter)*ys[j-1]
                ax2.plot(xs/xs[-1], ys, alpha=alpha, color=colors[file_index])

for a in axs:
    a.set_xlabel(r"$\frac{t-t_{min}}{t_{max}-t_{min}}$", fontsize=18)
axs[0].set_ylabel("Resistance (kOhm)", fontsize=15)
axs[1].set_ylabel("Current (A)", fontsize=15)
axs[2].set_ylabel("Capacitance (microF)", fontsize=15)


plot.show()

In [ ]:
extended_separated_transactions_status_faulted = {}
array_closest = []
for file_index in range(4):
    extended_separated_transactions_status_faulted[file_index] = []
    closest = pd.Timedelta('10d')
    closest_index = 0
    for status_list_index, status_list in enumerate(extended_separated_transactions_status[file_index]):
        temp = []
        for status in status_list:
            if status[1]["status"] == "Faulted":
                diff = np.abs(status[0].tz_localize(None)-times_of_explosion[file_index])
                if diff < closest:
                    closest = diff
                    closest_index = status_list_index
                
                temp.append(status)
        extended_separated_transactions_status_faulted[file_index].append(temp)
    array_closest.append(closest_index)

print(array_closest)
array_dics = []
for file_index in [0,1,2,3]:
    dic = defaultdict(list)
    for dc in [1,2]:
        if (file_index == 1 and dc == 2) or (file_index==0 and dc==2) or (file_index == 2 and dc == 1) or (file_index == 3 and dc==2):
            continue
        temp = []
        for index_resistance, resistances in enumerate(extended_separated_transactions_resitances[file_index][dc]):
            if len(resistances):
                x = resistances[-1][0].tz_localize(None)
                diff = times_of_explosion[file_index]-x if times_of_explosion[file_index]>x else x-times_of_explosion[file_index]
                temp.append([diff, index_resistance])
        
        temp = sorted(temp, key=lambda x: x[0])
        closest = temp[0][1]
        for index_alerts, alerts in enumerate(extended_separated_transactions_alert[file_index][dc]):
            if len(alerts):
                for alert in alerts:
                    x = alert[0].tz_localize(None)
                    diff = times_of_explosion[file_index]-x if times_of_explosion[file_index]>x else x-times_of_explosion[file_index]
                    dic[alert[1]["payload"]["msg"]].append([diff, index_alerts, closest])
    
    array_dics.append(dic)
